In [1]:
!nvidia-smi

Thu Oct 22 20:26:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!apt-get update -qq && apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 125 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (346 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
import subprocess
import time

ollama_log = open("/tmp/ollama.log", "w")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

time.sleep(5)
!curl http://localhost:11434

Ollama is running

In [7]:
!ollama pull qwen3:4b

In [8]:
!pip -q install ollama

In [10]:

import json
import ollama


def count_interpretations(interpretations: list[str]) -> dict:
    """Count the candidate interpretations generated by the model."""
    return {
        "interpretation_count": len(interpretations),
        "interpretations": interpretations,
    }


def main():
    sentence = input("Enter a sentence to analyze: ")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an interpretative openness research assistant. "
                "Identify up to three genuinely distinct, plausible "
                "interpretations of the user's sentence. Explain each "
                "briefly. Then call count_interpretations with your "
                "candidate interpretations. Do not invent interpretations "
                "that the sentence cannot reasonably support."
            ),
        },
        {
            "role": "user",
            "content": sentence,
        },
    ]

    # Ask the LLM to analyze the sentence and use our Python tool.
    response = ollama.chat(
        model="qwen3:4b",
        messages=messages,
        tools=[count_interpretations],
    )

    messages.append(response.message)

    # Execute any tool calls requested by the model.
    for tool_call in response.message.tool_calls or []:
        if tool_call.function.name == "count_interpretations":
            result = count_interpretations(
                **tool_call.function.arguments
            )

            messages.append({
                "role": "tool",
                "tool_name": "count_interpretations",
                "content": json.dumps(result),
            })

    # Let the model explain the result returned by Python.
    final_response = ollama.chat(
        model="qwen3:4b",
        messages=messages,
    )

    print("\n--- Interpretative Openness Explorer ---")
    print(final_response.message.content)


if __name__ == "__main__":
    main()


Enter a sentence to analyze: She saw her duck

--- Interpretative Openness Explorer ---
She saw her pet duck (a duck that she owns).  
She saw a small amount of money (slang for "duck" as a small sum).  
She saw the person who goes by the nickname "duck".  

{"name": "count_interpretations", "arguments": {"interpretations": [{"text": "She saw her pet duck (a duck that she owns)."}, {"text": "She saw a small amount of money (slang for 'duck' as a small sum)."}, {"text": "She saw the person who goes by the nickname 'duck'."}]}}
